In [12]:
import numpy as np
import pandas as pd
import cv2
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential #type:ignore
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D #type:ignore
from tensorflow.keras.optimizers import Adam #type:ignore
from tensorflow.keras.utils import to_categorical #type:ignore
from tensorflow.keras.preprocessing.image import ImageDataGenerator #type:ignore


In [13]:

path = "myData"  # Training dataset folder
labelFile = "labels.csv"  # CSV file containing class labels
imageDimensions = (32, 32, 1)
batch_size_val = 50
epochs_val = 10
steps_per_epoch_val = 200
testRatio = 0.2
validationRatio = 0.2

# STEP 1: Read images and labels

images = []
classNo = []

myList = os.listdir(path)
print("Total Classes Detected:", len(myList))
numberOfClasses = len(myList)

for x in range(0, numberOfClasses):
    myPicList = os.listdir(path + "/" + str(x))
    for y in myPicList:
        curImg = cv2.imread(path + "/" + str(x) + "/" + y)
        curImg = cv2.resize(curImg, (imageDimensions[0], imageDimensions[1]))
        images.append(curImg)
        classNo.append(x)

images = np.array(images)
classNo = np.array(classNo)


Total Classes Detected: 43


In [14]:
# STEP 2: Split Data

X_train, X_test, y_train, y_test = train_test_split(images, classNo, test_size=testRatio)
X_train, X_validation, y_train, y_validation = train_test_split(X_train, y_train,test_size=validationRatio)


In [15]:
# STEP 3: Preprocessing

def preprocessing(img):
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.equalizeHist(img)
    img = img / 255
    return img

X_train = np.array(list(map(preprocessing, X_train))).reshape(-1, imageDimensions[0], imageDimensions[1], 1)
X_validation = np.array(list(map(preprocessing, X_validation))).reshape(-1, imageDimensions[0], imageDimensions[1], 1)
X_test = np.array(list(map(preprocessing, X_test))).reshape(-1, imageDimensions[0], imageDimensions[1], 1)


In [16]:
# STEP 4: One Hot Encoding Labels

y_train = to_categorical(y_train, numberOfClasses)
y_validation = to_categorical(y_validation, numberOfClasses)
y_test = to_categorical(y_test, numberOfClasses)


In [17]:
# STEP 5: Data Augmentation

dataGen = ImageDataGenerator(width_shift_range=0.1,
                             height_shift_range=0.1,
                             zoom_range=0.2,
                             shear_range=0.1,
                             rotation_range=10)
dataGen.fit(X_train)


In [18]:
# STEP 6: Define CNN Model

def myModel():
    noOfFilters = 60
    sizeOfFilter = (5,5)
    sizeOfFilter2 = (3,3)
    sizeOfPool = (2,2)
    noOfNodes = 500

    model = Sequential()
    model.add(Conv2D(noOfFilters, sizeOfFilter, activation='relu', input_shape=(imageDimensions[0], imageDimensions[1], 1)))
    model.add(Conv2D(noOfFilters, sizeOfFilter, activation='relu'))
    model.add(MaxPooling2D(pool_size=sizeOfPool))

    model.add(Conv2D(noOfFilters//2, sizeOfFilter2, activation='relu'))
    model.add(Conv2D(noOfFilters//2, sizeOfFilter2, activation='relu'))
    model.add(MaxPooling2D(pool_size=sizeOfPool))
    model.add(Dropout(0.5))

    model.add(Flatten())
    model.add(Dense(noOfNodes, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(numberOfClasses, activation='softmax'))

    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = myModel()
print(model.summary())


c:\Users\Flex5\Desktop\open cv project\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 60)     │         1,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 60)     │        90,060 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 60)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 10, 10, 30)     │        16,230 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 8, 8, 30)       │         8,130 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 4, 4, 30)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4, 4, 30)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 480)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 500)            │       240,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 500)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 43)             │        21,543 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 378,023 (1.44 MB)

 Trainable params: 378,023 (1.44 MB)

 Non-trainable params: 0 (0.00 B)

None


In [19]:

# STEP 7: Train Model

history = model.fit(
    dataGen.flow(X_train, y_train, batch_size=batch_size_val),
    steps_per_epoch=steps_per_epoch_val,
    epochs=epochs_val,
    validation_data=(X_validation, y_validation),
    shuffle=True
)


Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 21s 92ms/step - accuracy: 0.1522 - loss: 3.1676 - val_accuracy: 0.4883 - val_loss: 1.9151
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 17s 87ms/step - accuracy: 0.4096 - loss: 2.0266 - val_accuracy: 0.7676 - val_loss: 0.8783
Epoch 3/10
 46/200 ━━━━━━━━━━━━━━━━━━━━ 11s 75ms/step - accuracy: 0.5179 - loss: 1.6403

c:\Users\Flex5\Desktop\open cv project\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.5135 - loss: 1.6342 - val_accuracy: 0.7619 - val_loss: 0.8345
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 17s 86ms/step - accuracy: 0.5758 - loss: 1.3881 - val_accuracy: 0.8761 - val_loss: 0.4947
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 16s 79ms/step - accuracy: 0.6499 - loss: 1.1288 - val_accuracy: 0.8971 - val_loss: 0.3488
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.6930 - loss: 1.0042 - val_accuracy: 0.8983 - val_loss: 0.3888
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 17s 83ms/step - accuracy: 0.7065 - loss: 0.9384 - val_accuracy: 0.9161 - val_loss: 0.2941
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 18s 89ms/step - accuracy: 0.7491 - loss: 0.8019 - val_accuracy: 0.9353 - val_loss: 0.2468
Epoch 9/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.7461 - loss: 0.7951 - val_accuracy: 0.9368 - val_loss: 0.2315
Epoch 10/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.7848 - loss: 0.6988 - val_accuracy:

In [20]:

# STEP 8: Save Model

model.save("cnn_model_improved.h5")
print("Model Saved Successfully → cnn_model_improved.h5")


Model Saved Successfully → cnn_model_improved.h5


In [21]:

# STEP 9: Evaluation Metrics

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

# Predict on test data
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

# Calculate Metrics
accuracy = accuracy_score(y_true, y_pred_classes)
precision = precision_score(y_true, y_pred_classes, average='macro')
recall = recall_score(y_true, y_pred_classes, average='macro')
f1 = f1_score(y_true, y_pred_classes, average='macro')

print("\n===== MODEL PERFORMANCE (CNN) =====")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)
print("\nConfusion Matrix:\n", cm)


218/218 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step

===== MODEL PERFORMANCE (CNN) =====
Accuracy : 0.9623563218390805
Precision: 0.9605585528302366
Recall   : 0.9388569951049068
F1 Score : 0.944264269879967

Confusion Matrix:
 [[ 24   6   0 ...   0   0   0]
 [  0 384   3 ...   0   0   0]
 [  0   6 380 ...   0   0   0]
 ...
 [  0   0   0 ...  56   0   0]
 [  0   0   0 ...   0  41   0]
 [  0   0   0 ...   0   0  40]]
